# `%%ai` 매직 + Jupyternaut 사용 데모

이 노트북은 `jupyter-ai-magics`의 주요 기능을 ramdump 분석 컨텍스트에서 보여줍니다.

## 두 가지 LLM 호출 경로

| 경로 | 호출 방법 | 모델 ID 형식 | 환경변수 |
|------|-----------|-------------|----------|
| Chat UI | `File > New > Chat` → `@Jupyternaut` | `openrouter/<model>` | `OPENROUTER_API_KEY` |
| 노트북 셀 | `%%ai nemo` | `openai-chat-custom:<model>` | `OPENAI_API_KEY` + `OPENAI_API_BASE` |

**시작 전**: `./scripts/start_jupyter.sh` 으로 JupyterLab을 실행했다면 두 환경변수가 자동 설정됩니다.

## Step 0. 매직 로드 및 provider 목록 확인

In [ ]:
%load_ext jupyter_ai_magics

In [ ]:
# 사용 가능한 provider 목록 확인
# 'openai-chat-custom' 항목에서 모든 모델 ID(*)가 허용됨을 확인
%ai list

## Step 1. 기본 호출 (alias 사용)

alias는 `~/.ipython/profile_default/ipython_config.py`에서 설정됩니다:
- `nemo` → `openai-chat-custom:nvidia/nemotron-3-super-120b-a12b`
- `deepseek` → `openai-chat-custom:deepseek/deepseek-chat`
- `claude35` → `openai-chat-custom:anthropic/claude-3.5-sonnet`

alias 없이 전체 ID를 쓰려면: `%%ai openai-chat-custom:nvidia/nemotron-3-super-120b-a12b`

In [ ]:
%%ai nemo
Android 커널 패닉(kernel panic)이 발생하는 주요 원인 3가지를 간결하게 설명해줘.

## Step 2. 출력 포맷 지정 (`-f` 플래그)

지원 포맷: `text`, `markdown`, `code`, `html`, `math`, `json`, `image`

In [ ]:
%%ai nemo -f code
Python으로 /proc/kmsg 로그 파일을 읽어 'panic' 키워드가 포함된 줄만 출력하는 함수를 작성해줘.

In [ ]:
%%ai nemo -f markdown
ramdump 분석 시 확인해야 할 핵심 항목들을 마크다운 체크리스트로 정리해줘.

## Step 3. 변수 보간 (Python 변수를 프롬프트에 주입)

`{변수명}` 형식으로 현재 셀 실행 환경의 Python 변수를 프롬프트에 삽입할 수 있습니다.
분석 결과 딕셔너리를 LLM에 그대로 전달할 때 유용합니다.

In [ ]:
# 분석 결과 예시 (실제 사용 시 memory_kernel_analyzer.py의 summarize_findings() 결과)
sample_findings = {
    "file_info": {"size_gb": 3.8, "format": "raw"},
    "panic_signals": ["kernel BUG at mm/slab.c:2887", "Unable to handle kernel NULL pointer"],
    "process_count": 312,
    "suspicious_processes": ["kworker/2:1H", "mali_jd_worker"],
    "network_anomalies": ["unexpected socket state: CLOSE_WAIT x47"]
}

In [ ]:
%%ai nemo -f markdown
아래 Android ramdump 분석 결과를 보고 가장 가능성 높은 root cause와 다음 조사 방향을 제안해줘.

분석 결과:
{sample_findings}

## Step 4. 셀 히스토리 보간

`{In[N]}`, `{Out[N]}`, `{Err[N]}`으로 이전 셀의 입력/출력/에러를 참조할 수 있습니다.

In [ ]:
# 이전 셀(sample_findings 정의)의 출력을 참조해 후속 질문
# In[N] 번호는 실제 실행 순서에 맞게 조정하세요
print(sample_findings)

In [ ]:
%%ai nemo
위 분석 결과에서 suspicious_processes 항목의 각 프로세스가 커널 패닉과 연관될 수 있는 이유를 설명해줘.

컨텍스트: {Out[-1]}

## Step 5. 에러 디버깅 (`%ai fix`)

`%ai fix <alias>` 는 직전 셀에서 발생한 Python 에러를 LLM에 전달해 설명과 수정 방법을 요청합니다.

In [ ]:
# 의도적인 에러 셀 (실행하면 TypeError 발생)
data = {"panic": ["BUG at slab.c", "NULL pointer"]}
print(data["panic"] + " extra text")  # list + str → TypeError

In [ ]:
# 위 에러를 LLM이 한국어로 설명하고 수정 방법 제안
%ai fix nemo

## Step 6. 컨텍스트 관리

- `max_history = 4`: 최근 4번의 대화 교환을 유지합니다 (`ipython_config.py`에서 설정)
- `%ai reset`: 대화 히스토리를 초기화합니다 (새 분석 주제로 전환 시)

In [ ]:
# 대화 히스토리 초기화 (새 분석 세션 시작)
%ai reset

## Step 7. Chat UI (Jupyternaut)와 연동

노트북에서 분석한 결과를 Chat UI에서 이어받아 대화형으로 질의할 수 있습니다.

### 방법 1: `@file:` 첨부
Chat UI에 파일 경로를 직접 첨부:
```
@Jupyternaut @file:notebooks/interactive_log_analyzer.ipynb 이 노트북의 분석 결과를 요약해줘
```

### 방법 2: 활성 셀 컨텍스트
노트북에서 분석 결과가 담긴 셀을 선택한 상태로 Chat UI에서:
```
@Jupyternaut 방금 분석한 panic_signals 중에서 가장 심각한 것을 골라줘
```
Jupyternaut의 `get_active_cell_id` 도구가 현재 선택된 셀을 자동으로 참조합니다.

### 방법 3: 노트북 직접 질의
```
@Jupyternaut notebooks/interactive_log_analyzer.ipynb 3번 셀의 코드를 설명해줘
```